# Avaliar previsões recebidas em ficheiros

Atividades do capítulo 14 e do apêndice A. Ligamos as respostas de referência às previsões por identificador. Não ajustamos modelos nem escolhemos um novo limiar.

Execute as células pela ordem apresentada. No Colab, use o ZIP completo dos materiais de apoio; no computador, mantenha este caderno na pasta principal. Guarde as suas interpretações em células de texto. As saídas são calculadas quando executar o caderno.


## Preparar os ficheiros


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

# No computador, abra este caderno junto de reproduzir.py.
# No Colab, a primeira execução pede o ZIP completo dos materiais.
candidatos = [Path.cwd(), Path.cwd().parent]
pasta_colab = Path('/content/materiais_biociencias')
if pasta_colab.exists():
    candidatos += [p.parent for p in pasta_colab.rglob('reproduzir.py')]
raiz = next((p for p in candidatos if (p / 'reproduzir.py').is_file()), None)
if raiz is None:
    try:
        from google.colab import files
    except ImportError as erro:
        raise FileNotFoundError(
            'Abra o caderno na pasta que contém reproduzir.py, dados e reproducao.'
        ) from erro
    carregados = files.upload()
    arquivos = [Path(nome) for nome in carregados if nome.lower().endswith('.zip')]
    if len(arquivos) != 1:
        raise ValueError('Carregue um único ZIP completo dos materiais de apoio.')
    pasta_colab.mkdir(parents=True, exist_ok=True)
    destino = pasta_colab.resolve()
    with zipfile.ZipFile(arquivos[0]) as arquivo:
        for item in arquivo.infolist():
            caminho = (destino / item.filename).resolve()
            if not caminho.is_relative_to(destino):
                raise ValueError('O ZIP contém um caminho fora da pasta de destino.')
        arquivo.extractall(destino)
    encontrados = list(destino.rglob('reproduzir.py'))
    if len(encontrados) != 1:
        raise ValueError('O ZIP deve conter uma única pasta completa dos materiais.')
    raiz = encontrados[0].parent
os.chdir(raiz)
print('Pasta de trabalho:', raiz.name)

def executar_python(script, *argumentos):
    subprocess.run([sys.executable, script, *argumentos], check=True, cwd=raiz)


## Conferir o ambiente

Esta célula instala as dependências apenas se necessário. Se pedir um reinício, reinicie o ambiente e volte à primeira célula.


In [ ]:
from importlib.metadata import version, PackageNotFoundError

requisitos = {}
for linha in Path('requisitos.txt').read_text().splitlines():
    if '==' in linha and not linha.lstrip().startswith('#'):
        nome, esperada = linha.strip().split('==', 1)
        requisitos[nome] = esperada
alterar = False
for nome, esperada in requisitos.items():
    try:
        atual = version(nome)
    except PackageNotFoundError:
        atual = 'não instalada'
    print(f'{nome}: {atual} (referência: {esperada})')
    alterar = alterar or atual != esperada
if alterar:
    ja_importadas = any(n in sys.modules for n in ['numpy', 'pandas', 'sklearn', 'matplotlib', 'scipy'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requisitos.txt'], check=True)
    if ja_importadas:
        raise RuntimeError(
            'Dependências instaladas. Reinicie o ambiente e execute desde a primeira célula '
            'para usar as versões agora instaladas.'
        )
else:
    print('O ambiente já tem as versões de referência.')


## 1. Inspecionar as seis amostras fictícias

As linhas estão em ordens diferentes. Os dados são didáticos e não pertencem a Wisconsin. Leia também `exemplo_previsoes/NOTA_DO_PROCEDIMENTO.md`.


In [ ]:
import pandas as pd
from IPython.display import display, Image
pasta = Path('exemplo_previsoes')
display(pd.read_csv(pasta / 'referencia_avaliacao.csv', dtype={'id_registo': 'string'}))
display(pd.read_csv(pasta / 'previsoes_avaliacao.csv', dtype={'id_registo': 'string'}))


## 2. Verificar a correspondência e avaliar

O programa recusa IDs repetidos, em falta ou inesperados. Conserva as classes previstas recebidas. As medidas sem denominador positivo ficam assinaladas como não definidas.


In [ ]:
executar_python('reproducao/previsoes/avaliar_previsoes.py', 'classificacao')


In [ ]:
saida = pasta / 'resultados'
display(pd.read_csv(saida / 'erros_por_amostra.csv'))
display(pd.read_csv(saida / 'medidas.csv'))
display(Image(filename=str(saida / 'matriz_contagens.png')))
display(Image(filename=str(saida / 'roc.png')))
display(Image(filename=str(saida / 'vpp_sensibilidade.png')))


## 3. Usar as exportações reais Abalone

Estas previsões foram produzidas pelo estudo do capítulo 14. A ligação reconstitui a mesma avaliação; não é uma segunda validação independente.


In [ ]:
executar_python('reproducao/previsoes/avaliar_previsoes.py', 'abalone')


In [ ]:
saida_abalone = Path('reproducao/previsoes/resultados_abalone')
display(pd.read_csv(saida_abalone / 'medidas.csv'))
display(pd.read_csv(saida_abalone / 'erros_ordenados.csv').head())


## Interpretar e guardar

Confirme A03 como falso positivo e A04 como falso negativo. Explique por que a ordem das linhas não define a correspondência. Para usar outros ficheiros, consulte os argumentos do script no README do percurso e registe a origem das respostas, o procedimento e os IDs esperados. Guarde os resultados antes de terminar a sessão Colab.
